In [1]:
!pip install -q langchain-core requests langchain-ollama

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [3]:
llm = ChatOllama(
    model="qwen3:4b",
    temperature=0
)

## tool Create : 

In [5]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/e2bc0b4339cd0cee8c03e78d/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [6]:
get_conversion_factor.invoke({'base_currency' : 'USD' , 'target_currency' : 'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1789776001,
 'time_last_update_utc': 'Sat, 19 Sep 2026 00:00:01 +0000',
 'time_next_update_unix': 1789862401,
 'time_next_update_utc': 'Sun, 20 Sep 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.9935}

In [7]:
convert.invoke({'base_currency_value' : 10 , 'conversion_rate' : 95.9935})

959.935

## tool binding : 

In [8]:
llm_with_tools = llm.bind_tools([get_conversion_factor , convert])

In [9]:
llm_with_tools

_ChatModelBinding(bound=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0'}}, model='qwen3:4b', temperature=0.0), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_conversion_factor', 'description': 'This function fetches the currency conversion factor between a given base currency and a target currency', 'parameters': {'properties': {'base_currency': {'type': 'string'}, 'target_currency': {'type': 'string'}}, 'required': ['base_currency', 'target_currency'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'convert', 'description': 'given a currency conversion rate this function calculates the target currency value from a given base currency value', 'parameters': {'properties': {'base_currency_value': {'type': 'integer'}}, 'required': ['base_currency_value'], 'type': 'object'}}}]}, config={}, config_factories=[])

## tool Calling : 

In [10]:
messages = [HumanMessage('What is the conversion factor between USD and INR , and based on that can you convert 10 USD to INR')]

In [11]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR , and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]